In [ ]:
import torch
import torchvision
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from matplotlib import pyplot as plt
from GhostNet import GhostNet
from ghost_simple import SimpleGhost
from GhostNet_change import GhostNet as GhostNet_change

KeyboardInterrupt: 

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed = 100
set_seed(seed)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307, ), (0.3081, )),
    transforms.Grayscale(1)
])

In [ ]:
mnist_db = datasets.MNIST(root= 'dataFolder/', train= True, transform= transform, download= True)
mnist_loader = DataLoader(mnist_db, batch_size= 64, shuffle= True)

In [ ]:
def to_rgb(x):
    if x.shape[1] == 3: return x
    elif x.shape[1] == 1: return x.reapeat_interleave(3, dim= 1)
    elif x.shape[1] == 4: return x[:, :3, :, :]
    else: raise ValueError('Invalid input shape')

tensor_rgb_transform = transforms.Lambda(to_rgb)

In [ ]:
def train_model(model, train_loader, in_ch):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(device)

    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr= 0.001)
    model.train()

    state_dict = None
    max_acc = 0
    min_loss = 1
    losses = []
    for epoch in range(10):  # 训练10个周期
        correct = 0
        total = 0
        for idx, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)
            if in_ch == 3: x = tensor_rgb_transform(x)
            out = model(x)
            loss = criterion(out, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += x.shape[0]
            correct += (torch.argmax(out, dim= -1) == y).sum().item()
            losses.append(loss.item())
            if idx % 1000 == 0:
                print('Epoch: {}, Loss: {}'.format(epoch, loss.item()))
                print('Accuracy: {:.2f}%'.format(correct * 100 / total))
        if correct / total >= max_acc or min_loss > losses[-1]:
            max_acc = correct / total
            min_loss = losses[-1]
            state_dict = model.state_dict()

    return state_dict, losses

In [ ]:
model = SimpleGhost(in_channels= 1)
state_dict, losses = train_model(model, mnist_loader, 1)
torch.save(state_dict, f'models/simple/ghostnet_simple_md_{seed}.pth')

cuda
Epoch: 0, Loss: 2.321869134902954
Accuracy: 10.94%
Epoch: 1, Loss: 0.0850241407752037
Accuracy: 96.88%
Epoch: 2, Loss: 0.022357702255249023
Accuracy: 100.00%
Epoch: 3, Loss: 0.002094930037856102
Accuracy: 100.00%
Epoch: 4, Loss: 0.0005520472768694162
Accuracy: 100.00%
Epoch: 5, Loss: 0.022676821798086166
Accuracy: 98.44%
Epoch: 6, Loss: 0.07385613024234772
Accuracy: 95.31%
Epoch: 7, Loss: 0.0033231086563318968
Accuracy: 100.00%
Epoch: 8, Loss: 0.00022349301434587687
Accuracy: 100.00%
Epoch: 9, Loss: 0.0008620898006483912
Accuracy: 100.00%


In [ ]:
# block_types = ['inverted_residual', 'shuffle', 'ghost', 'residual', 'basic', 'selayer'] # 
# for block_type in block_types:
#     model = PrecisionBalancedCNN(block_type, in_channels= 1)
#     state_dict, losses = train_model(model, mnist_loader, 1)
#     torch.save(state_dict, f'models/2/model_{block_type}_2_{seed}.pth')
#     print('model train success')